# Cascade prototype
SW7 project, group S7G5, AAU Copenhagen.

The prototype follows this route:

1. High-specificity rule-based block.
2. Char TF-IDF + calibrated SVM block/pass/unsure decision.
3. Prompt Guard 2 only for unsure prompts.

Database TODO: add the shared signature/embedding lookup before the rule tier.
Database TODO: store Prompt Guard 2-confirmed attacks once the shared database is implemented.

Results are saved in `results/cascade/`.

## 1. Setup
Run this notebook at the repository root. In Colab, the first cell clones the repository and moves into `classifier-testing`.

In [ ]:
import os
from pathlib import Path

if "COLAB_RELEASE_TAG" in os.environ:
    if not Path("Software7Group5").exists():
        !git clone https://github.com/KonDiamanti/Software7Group5.git
    %cd Software7Group5/classifier-testing

%pip install -q -r requirements.txt
%pip install -q sentence-transformers

try:
    %load_ext autoreload
    %autoreload 2
except Exception:
    pass

In [ ]:
import sys
import time
import pandas as pd

from huggingface_hub import login, get_token

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent / "core-skeleton"))

from cascade import Cascade, evaluate, summary, lazy_prompt_guard
from prompt_data import load_test_prompts, load_training_data
print("Hugging Face login:", "yes" if get_token() else "no")

For gated HackAPrompt and Prompt Guard 2 access, run the next cell. Login is optional when testing public datasets and with `USE_PROMPT_GUARD=False`.

In [ ]:
# login()
USE_PROMPT_GUARD = True
USE_HACKAPROMPT = True
HACKAPROMPT_SAMPLE = 500
# Database is deliberately excluded from this prototype run.
# TODO: insert a shared signature/embedding lookup before the rule tier.

## 2. Data
The test set contains deepset, jackhhao, optional HackAPrompt, and our own prompts. Training prompts are drawn from safe-guard after removing test near-copies.

In [ ]:
data = load_test_prompts(HACKAPROMPT_SAMPLE if USE_HACKAPROMPT else 0)
display(data.groupby(["dataset", "label"], sort=False).size().unstack(fill_value=0)
        .rename(columns={0: "harmless", 1: "attacks"}))

safe_guard = load_training_data(data["text"])
train = safe_guard.to_pandas() if hasattr(safe_guard, "to_pandas") else safe_guard

## 4. Guard and cascade
Prompt Guard 2 is loaded only if a prompt reaches the uncertain tier.

In [ ]:
guard = lazy_prompt_guard() if USE_PROMPT_GUARD else None
if guard is None:
    print("Prompt Guard 2 is disabled; uncertain prompts stay 'uncertain'.")

cascade = Cascade(guard_check=guard)
training_seconds = cascade.train(train)
print("Char TF-IDF + SVM training seconds:", round(training_seconds, 3))
print("Block cutoff:", round(cascade.block_cutoff, 4))
print("Pass cutoff:", round(cascade.pass_cutoff, 4))

## 5. Run and save results

In [ ]:
RESULT_DIR = ROOT / "results" / "cascade"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

predictions = evaluate(cascade, data)
predictions.to_csv(RESULT_DIR / "cascade_predictions.csv", index=False)

results = summary(predictions)
results.to_csv(RESULT_DIR / "cascade_results.csv", index=False)
display(results.round(4))

In [ ]:
stage_counts = predictions.groupby(["dataset", "stage"], sort=False).size().unstack(fill_value=0)
display(stage_counts.fillna(0).astype(int))

missed_attacks = int(((predictions["label"] == 1) & (predictions["decision"] != "block")).sum())
false_alarm_count = int(((predictions["label"] == 0) & (predictions["decision"] == "block")).sum())

print("Missed attacks:", missed_attacks)
print("False alarms:", false_alarm_count)